[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S07/S07_01_regresion_lineal_polinomica.ipynb)

# S07 · 01 — Regresión lineal y polinómica: predecir el precio por noche

**Tiempo estimado:** 60 min · **Sprint 7** · M3 (UD 3.3)

**Contexto TurisData.** El cliente quiere una tarifa de referencia: *¿cuánto debería costar
la noche de una reserva según el tipo de habitación y la época?* Es un problema de
**regresión** (predecir un número). Empezamos por el modelo más sencillo y más explicable:
la regresión lineal.

**Al terminar sabrás:**
1. Ajustar una regresión lineal e **interpretar sus coeficientes en euros**.
2. Medir el error (MAE, RMSE, R²) y mirar los **residuos**.
3. Usar términos polinómicos y de interacción y saber cuándo compensan.
4. Comparar con un baseline y con una variable bien construida (*feature engineering*).

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["mes_llegada"] = df["fecha_llegada"].dt.month
df["temporada_alta"] = df["mes_llegada"].isin([1, 2, 3, 7, 8, 12]).astype(int)
print(df.shape)

## 1. La idea: una recta (o un plano) que se ajusta a los datos

La regresión lineal supone que el precio es una **suma ponderada** de las variables:

`precio ≈ base + (efecto de suite)·es_suite + (efecto de temporada alta)·temporada_alta + …`

Cada **coeficiente** dice *cuántos euros cambia el precio al aumentar esa variable en una
unidad, manteniendo las demás iguales*. Por eso es tan apreciada: se puede **explicar**.
Las categorías (tipo de habitación) se convierten en columnas 0/1 (*dummies*).

In [ ]:
X = pd.get_dummies(df[["tipo_habitacion"]], drop_first=True).astype(int)
X["temporada_alta"] = df["temporada_alta"]
y = df["precio_noche"]
X.head()

`drop_first=True` quita una categoría (`estandar`), que actúa como **referencia**: los
coeficientes de las otras se leen como diferencia respecto a la habitación estándar.

**Ejercicio 1.** Separa en `X_train, X_test, y_train, y_test` (20 % de test, `random_state=SEMILLA`),
ajusta una `LinearRegression` llamada `modelo_lin` y guarda en `r2_test` y `mae_test` sus
métricas en test.

In [ ]:
# TODO: train_test_split(X, y, test_size=0.2, random_state=SEMILLA); LinearRegression().fit(...)
...
assert len(X_test) == 1200, "Con 6000 filas y test_size=0.2 el test debe tener 1200 filas"
assert r2_test > 0.85, "R² debería superar 0,85: revisa que uses tipo de habitación y temporada"
assert 5 < mae_test < 15, "El MAE debería estar entre 5 y 15 euros"
print(f"R² = {r2_test:.3f} · MAE = {mae_test:.2f} € · RMSE = {mean_squared_error(y_test, pred) ** 0.5:.2f} €")

### ¿Mejor que "no hacer nada"?
El **baseline** de regresión predice siempre la media del precio.

In [ ]:
base = DummyRegressor(strategy="mean").fit(X_train, y_train)
print(f"Baseline: MAE = {mean_absolute_error(y_test, base.predict(X_test)):.2f} € · R² = {r2_score(y_test, base.predict(X_test)):.3f}")

## 2. Interpretar los coeficientes

**Ejercicio 2.** Consulta `modelo_lin.coef_` (van en el mismo orden que las columnas de `X`).
Guarda en `coefs` un `pd.Series` con los coeficientes y el nombre de columna como índice,
y responde con números: `extra_suite` (euros extra de una suite frente a la estándar)
y `efecto_temporada_alta` (euros extra por reservar en temporada alta).

In [ ]:
# TODO: pd.Series(modelo_lin.coef_, index=X.columns)
...
assert 90 < extra_suite < 115, "Una suite cuesta unos 100 € más que una estándar (coeficiente de tipo_habitacion_suite)"
assert 15 < efecto_temporada_alta < 30, "La temporada alta añade unos 20-25 €"
print(coefs.round(1))
print("Ordenada en el origen (precio base de una estándar en temporada baja):", round(modelo_lin.intercept_, 1), "€")

## 3. Residuos: ¿el modelo se equivoca de forma "sana"?

El **residuo** es `real − predicho`. Si el modelo está bien planteado, los residuos se
reparten **sin patrón** alrededor de 0. Si vemos curvas o embudos, falta algo en el modelo.

In [ ]:
residuos = y_test - pred
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].scatter(pred, residuos, s=10, alpha=0.4, color="#2a6f97")
ejes[0].axhline(0, color="black", lw=1)
ejes[0].set(title="Residuos frente a predicción", xlabel="Precio predicho (€)", ylabel="Residuo (€)")
sns.histplot(residuos, bins=30, ax=ejes[1], color="#2a6f97")
ejes[1].set(title="Distribución de los residuos", xlabel="Residuo (€)", ylabel="Reservas")
plt.tight_layout()
plt.show()

Se ve un **embudo**: los errores crecen con el precio (la suite falla más, en euros, que la
estándar). Tiene sentido: en estos datos las variaciones de precio son **proporcionales**
(la temporada alta multiplica ×1,25 y el "ruido" es un % del precio), pero el modelo
lineal **suma** efectos fijos en euros. Una parte se puede corregir con **interacciones**
(que el efecto de la temporada dependa del tipo de habitación).

## 4. Términos polinómicos y de interacción

**Idea.** `PolynomialFeatures` crea columnas nuevas: potencias (`x²`, `x³`) y productos entre
variables (`x1·x2`, las **interacciones**). El modelo sigue siendo lineal *en los
coeficientes*, pero puede dibujar curvas. Con `interaction_only=True` solo crea productos.

In [ ]:
lineal = LinearRegression()
con_interacciones = make_pipeline(PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
                                  LinearRegression())
r2_lineal_cv = cross_val_score(lineal, X, y, cv=5, scoring="r2").mean()
r2_inter_cv = cross_val_score(con_interacciones, X, y, cv=5, scoring="r2").mean()
print(f"R² CV lineal: {r2_lineal_cv:.3f} · con interacciones: {r2_inter_cv:.3f}")

**Ejercicio 3.** Calcula `mae_inter_test`: el MAE en test del modelo `con_interacciones`
ajustado con `X_train, y_train`. Guarda en `mejora_mae` cuántos euros mejora respecto a
`mae_test` (positivo = mejor).

In [ ]:
# TODO: ajusta con_interacciones con train y predice sobre test
...
assert mae_inter_test < mae_test, "Las interacciones deberían reducir el error"
assert mejora_mae > 0.3, "La mejora debería ser apreciable (más de 0,3 €)"

## 5. Polinomios para captar curvas... y su peligro

Ahora estudiamos solo las habitaciones **estándar** y cómo cambia el precio con el **mes**
(variable numérica de 1 a 12). El precio sube en invierno y verano y baja en primavera y otoño:
una relación con curvas. Un polinomio de grado bajo no la capta; uno de grado alto sí, pero
se vuelve complejo y **puede sobreajustar**.

In [ ]:
est = df[df["tipo_habitacion"] == "estandar"]
X_mes, y_est = est[["mes_llegada"]], est["precio_noche"]
grados = list(range(1, 13))
r2_por_grado = []
for g in grados:
    m = make_pipeline(PolynomialFeatures(g), StandardScaler(), LinearRegression())
    r2_por_grado.append(cross_val_score(m, X_mes, y_est, cv=5, scoring="r2").mean())

mes_ord = pd.DataFrame({"mes_llegada": range(1, 13)})
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.scatter(est["mes_llegada"], est["precio_noche"], s=8, alpha=0.3, color="gray", label="Reservas estándar")
for g, c in [(2, "#e63946"), (8, "#2a6f97")]:
    m = make_pipeline(PolynomialFeatures(g), StandardScaler(), LinearRegression()).fit(X_mes, y_est)
    plt.plot(mes_ord, m.predict(mes_ord), color=c, lw=2, label=f"Grado {g}")
plt.title("Precio de la estándar según el mes")
plt.xlabel("Mes de llegada")
plt.ylabel("Precio por noche (€)")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(grados, r2_por_grado, marker="o", color="#2a6f97")
plt.title("R² (validación cruzada) según el grado")
plt.xlabel("Grado del polinomio")
plt.ylabel("R²")
plt.tight_layout()
plt.show()

**Ejercicio 4.** Compara con validación cruzada (5 folds, `scoring="r2"`) sobre las habitaciones
estándar dos formas de usar la época del año y guarda las medias en:
- `r2_grado2`: polinomio de grado 2 sobre `mes_llegada` (con `make_pipeline(PolynomialFeatures(2), LinearRegression())`),
- `r2_alta`: regresión lineal simple sobre la variable `temporada_alta` (una sola columna).

In [ ]:
# TODO: cross_val_score(modelo, X, y, cv=5, scoring="r2").mean() dos veces con distintos X
...
assert r2_alta > r2_grado2, "La variable temporada_alta debería rendir más que un polinomio de grado 2 sobre el mes"
assert r2_alta > 0.5, "temporada_alta debería explicar más de la mitad de la variación del precio de la estándar"
print(f"Grado 2 sobre el mes: R² = {r2_grado2:.2f} · temporada_alta (1 columna): R² = {r2_alta:.2f}")

**Lección:** pensar bien las variables (*feature engineering*: aquí «temporada alta» en lugar
del mes crudo) casi siempre supera a añadir complejidad al modelo. Un polinomio de grado alto
imita la curva, pero es más frágil y difícil de explicar.

## 6. Mini-reto: tarifa de referencia explicada al cliente

Con `modelo_lin` (o `con_interacciones`), prepara la función `precio_recomendado(tipo, alta)`
que devuelva el precio previsto (€) para un tipo de habitación (`"estandar"`, `"superior"`,
`"suite"`, `"familiar"`) y un valor de `alta` (0 o 1). Usa `modelo_lin`.

In [ ]:
# TODO: construye una fila con las mismas columnas que X (mismo orden) y usa modelo_lin.predict
...
p_std_baja = precio_recomendado("estandar", 0)
p_suite_alta = precio_recomendado("suite", 1)
assert 60 < p_std_baja < 80, "Una estándar en temporada baja ronda 70 €"
assert p_suite_alta > 180, "Una suite en temporada alta debería superar los 180 €"
print(f"Estándar en baja: {p_std_baja:.0f} € · Suite en alta: {p_suite_alta:.0f} €")

## 7. Para reflexionar

1. El coeficiente de `temporada_alta` es ≈ 22 €. ¿Por qué no es el mismo "extra" para una estándar que para una suite?
2. ¿Qué te dice el embudo en los residuos? ¿Cómo lo corregirías (interacciones, transformar el objetivo con logaritmo…)?
3. Un polinomio de grado 12 tiene R² alto en train. ¿Qué mirarías para saber si sobreajusta?
4. ¿Qué información **no** está en el modelo y que un revenue manager usaría para fijar precios (competencia, ocupación, eventos…)?

## 8. Resumen: qué has aprendido
- Regresión lineal = suma ponderada; los coeficientes se leen en unidades del objetivo.
- MAE (€), RMSE y R² miden el error; el baseline (media) da la referencia.
- Los residuos revelan lo que el modelo no capta; las interacciones y los polinomios lo mejoran.
- Una variable bien pensada vence a más complejidad.